# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [10]:
%pip -q install duckdb huggingface_hub


In [11]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [15]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [16]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [17]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [18]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [19]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.548     0.336     0.417      9389
           1      0.685     0.839     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.588     0.585     25551
weighted avg      0.635     0.654     0.630     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


**a. What one row means for my lane**
One row = one content item's Search Console performance on one day
(client_hash_id + content_hash_id + report_date) in fact_content_daily_performance.
My unit of analysis is one content item, as of a decision day.

**b. Which table(s) I use**
fact_content_daily_performance, sliced to month=2026-03. I leave fact_query_90d
out of this notebook — its 90-day trailing window isn't confirmed to align with
"as of March 15," so including it here risks a window-alignment leak (see limitation).

**c. Time window**
March 2026, split at the midpoint: days 1–15 = feature window (decision moment),
days 16–31 = outcome window the label is drawn from.

**d. What I'd predict or rank**
is_declining = 1 if summed impressions in days 16–31 < 80% of summed impressions
in days 1–15, else 0 — same proxy as notebook 02's trend_direction == 'down'.

**e. One thing I deliberately exclude**
fact_daily_sample (June 2026, the sealed final month) — never touched for
features or labels, only for a true holdout later.

3)Three queries, five features, the trap

Query 1 — grain **bold text**

In [20]:
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 1
""").df().iloc[0]

grain_check = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_on_this_date
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""").df()

print(f"rows in March: {len(grain_check)}")
print(f"max rows on any single date: {grain_check['rows_on_this_date'].max()}  (should be 1)")
grain_check.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in March: 31
max rows on any single date: 1  (should be 1)


,report_date,rows_on_this_date
0,2026-03-01,1
1,2026-03-02,1
2,2026-03-03,1
3,2026-03-04,1
4,2026-03-05,1
5,2026-03-06,1
6,2026-03-07,1
7,2026-03-08,1
8,2026-03-09,1
9,2026-03-10,1


**Query 2 — row count and date span**

In [21]:
slice_stats = con.sql(f"""
    SELECT COUNT(*)                         AS n_rows,
           COUNT(DISTINCT content_hash_id)  AS n_content_items,
           COUNT(DISTINCT client_hash_id)   AS n_clients,
           MIN(report_date)                 AS first_date,
           MAX(report_date)                 AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

slice_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


**Query 3 — availability with IS TRUE**

In [22]:
# Find the boolean column(s) actually in your schema first:
desc_daily = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
print(desc_daily[['column_name', 'column_type']].to_string(index=False))
bool_cols = desc_daily.loc[desc_daily['column_type'].str.upper() == 'BOOLEAN', 'column_name'].tolist()
print(f"\nBoolean columns found: {bool_cols}")

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

In [24]:
for name, src in TABLES.items():
    d = con.sql(f"DESCRIBE SELECT * FROM {src} LIMIT 0").df()
    bools = d.loc[d['column_type'].str.upper() == 'BOOLEAN', 'column_name'].tolist()
    if bools:
        print(f"{name}: {bools}")

dim_clients: ['is_active', 'has_gsc_access', 'has_ga4_access']
dim_content: ['is_published', 'is_deleted']
fact_daily: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
fact_daily_sample: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']


In [25]:
BOOLEAN_COLUMN = 'gsc_data_available'

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE {BOOLEAN_COLUMN} IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

availability['pct_available'] = (availability['available_rows'] / availability['total_rows'] * 100).round(1)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.7


**Five features (days 1–15 only)**

In [26]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS impressions_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_h1,
        AVG(gsc_avg_position)                                           AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

print(f'{len(feature_frame):,} content items with a usable feature row')
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with a usable feature row


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,0.004662,4.247255,15
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,0.000000,4.939394,11
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,0.000000,3.010741,15
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,0.001592,5.330069,15
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,0.007031,4.468441,15


1. impressions_h1 — sum of daily impressions, days 1–15. Knowable same-day, nothing from days 16–31.
2. clicks_h1 — sum of daily clicks, days 1–15. Same reasoning.
3. ctr_h1 — clicks_h1 / impressions_h1. Ratio of two already-knowable sums.
4. avg_position_h1 — mean daily rank position, days 1–15. Logged per report day.
5. active_days_h1 — count of days with impressions > 0 in the window. Just a count of days already past.

**The trap**

In [27]:
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(outcome, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining'] = (data['impressions_h2'] < 0.8 * data['impressions_h1']).astype(int)
print(f'{len(data):,} rows | declining rate: {round(data["is_declining"].mean(), 3)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 rows | declining rate: 0.296


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_h1', 'clicks_h1', 'ctr_h1', 'avg_position_h1', 'active_days_h1']
model_data = data.dropna(subset=honest_features + ['is_declining'])
X, y = model_data[honest_features], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'HONEST score — AUC: {honest_auc:.3f}')

HONEST score — AUC: 0.581


In [29]:
# THE TRAP: add the label-derived column
leaky_features = honest_features + ['impressions_h2']
leaky_data = data.dropna(subset=leaky_features + ['is_declining'])
X_leak, y_leak = leaky_data[leaky_features], leaky_data['is_declining']
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)

leaky_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f'LEAKY score — AUC: {leaky_auc:.3f}  <- jumps toward 1.0')
print(f'Jump: +{leaky_auc - honest_auc:.3f}')

LEAKY score — AUC: 1.000  <- jumps toward 1.0
Jump: +0.419


In [30]:
# REMOVE THE LEAK, KEEP THE HONEST NUMBER
del leaky_features
print(f'Final reported score (honest features only): AUC {honest_auc:.3f}')
print('impressions_h2 excluded — it is the quantity the label threshold is built from.')

Final reported score (honest features only): AUC 0.581
impressions_h2 excluded — it is the quantity the label threshold is built from.


**4) Named limitation**

Unbalanced panel coverage: dim_clients.gsc_data_start varies per client, so March 2026
isn't equally "mid-panel" for everyone in this slice — recently-onboarded clients get
thinner, noisier 15-day features than long-tenured ones, and this notebook doesn't
correct for that yet.

**5) Self-check**

- [ ] Five plain-words contract answers
- [ ] Three verification queries, outputs visible (availability via IS TRUE)
- [ ] Five-feature frame, one "available when?" line each
- [ ] Leak experiment shown, then removed, honest score kept
- [ ] One named limitation
- [ ] No client names/URLs in the notebook
- [ ] fact_daily_sample never touched for features/labels
- [ ] Runs top to bottom, no errors